In [ ]:
!pip uninstall -y matplotlib numpy
!pip install --user numpy==1.25.2 matplotlib==3.7.3


In [ ]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import os
import glob

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, LSTM, TimeDistributed, GRU
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
# from tensorflow.keras.preprocessing.sequence import pad_sequences # Not used for now

# --- Configuration ---
BASE_DATA_DIRECTORY = '../data' # UPDATE YOUR DATA PATH
FRAME_ID_COLUMN = 'timestamp_mmwave'
NUM_CLASSES = 6

THRESHOLD_INSIDE_TOF = 140
REGION_WINDOW_SIZE_FRAMES = 3

# mmWave X-Y Heatmap Configuration
X_MIN_METERS = -0.5; X_MAX_METERS = 0.5
Y_MIN_METERS = 0.1; Y_MAX_METERS = 1.0
X_BINS = 48; Y_BINS = 48
NUM_CHANNELS_XY_VEL = 2 # Channel 0: SNR/Count, Channel 1: Velocity

IMG_HEIGHT_XY = Y_BINS
IMG_WIDTH_XY = X_BINS

# Time Series Configuration
SEQUENCE_LENGTH = 5

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

# --- Data Discovery and Loading Functions ---
def discover_data_files(base_dir):
    discovered_paths_map = {}
    base_dir = os.path.normpath(base_dir)
    if not os.path.isdir(base_dir): print(f"Error: Base data directory '{base_dir}' not found."); return discovered_paths_map
    search_pattern = os.path.join(base_dir, '**', 'merged_df.csv')
    found_files = sorted(list(glob.glob(search_pattern, recursive=True)))
    if not found_files: print(f"Warning: No files found with '{search_pattern}' under '{base_dir}'."); return discovered_paths_map
    file_counter = 0
    for filepath in found_files:
        try:
            relative_path = os.path.relpath(filepath, base_dir)
            path_parts = relative_path.split(os.sep)
            experiment_type = path_parts[-2] if len(path_parts) > 1 and path_parts[-1] == 'merged_df.csv' else f"untyped_experiment_{file_counter}"
            if experiment_type == '.': experiment_type = f"untyped_experiment_{file_counter}"
            file_counter +=1; normalized_filepath = os.path.normpath(filepath)
            if experiment_type not in discovered_paths_map: discovered_paths_map[experiment_type] = []
            discovered_paths_map[experiment_type].append(normalized_filepath)
        except Exception as e: print(f"  Error while processing '{filepath}': {e}")
    if not discovered_paths_map: print(f"Warning: Data file could not be mapped from directory '{base_dir}'.")
    else: print(f"\nSuccessfully discovered data files: {len(discovered_paths_map)} experiment type(s).")
    return discovered_paths_map

def load_and_preprocess_data(data_paths_map_with_lists, frame_id_col_name):
    all_dataframes_merged = []
    for experiment_type, path_list in data_paths_map_with_lists.items():
        dataframes_for_current_type = []
        for path in path_list:
            if os.path.exists(path):
                try:
                    df = pd.read_csv(path); df['experiment_type'] = experiment_type
                    dataframes_for_current_type.append(df)
                except Exception as e: print(f"  Error while loading '{path}': {e}")
        if dataframes_for_current_type: all_dataframes_merged.append(pd.concat(dataframes_for_current_type, ignore_index=True))
    if not all_dataframes_merged: raise FileNotFoundError("No valid data file was loaded.")
    final_merged_df = pd.concat(all_dataframes_merged, ignore_index=True)
    print(f"Final merged (point level) data shape: {final_merged_df.shape}")
    datetime_cols_to_convert = list(set(['timestamp_mmwave', 'timestamp_dist', 'timestamp', frame_id_col_name]))
    for col in datetime_cols_to_convert:
        if col in final_merged_df.columns:
            if not pd.api.types.is_datetime64_any_dtype(final_merged_df[col]):
                try:
                    if pd.api.types.is_numeric_dtype(final_merged_df[col]): final_merged_df[col] = pd.to_datetime(final_merged_df[col], unit='s', errors='coerce')
                    else: final_merged_df[col] = pd.to_datetime(final_merged_df[col], errors='coerce')
                except Exception as e: print(f"Warning: '{col}' could not be converted to datetime: {e}.")
    if frame_id_col_name in final_merged_df.columns and pd.api.types.is_datetime64_any_dtype(final_merged_df[frame_id_col_name]):
        final_merged_df = final_merged_df.sort_values(by=[frame_id_col_name, 'experiment_type']).reset_index(drop=True)
    elif frame_id_col_name in final_merged_df.columns:
        final_merged_df = final_merged_df.sort_values(by=[frame_id_col_name, 'experiment_type']).reset_index(drop=True)
        print(f"Warning: '{frame_id_col_name}' is not datetime, sorted numerically.")
    else: print(f"WARNING: '{frame_id_col_name}' cannot be used for sorting.")
    return final_merged_df

# --- mmWave X-Y Heatmap Generation (VELOCITY CHANNEL ADDED) ---
def create_xy_heatmap_with_velocity(frame_points_df, snr_feature='snr', velocity_feature='doppler'):
    heatmap = np.zeros((Y_BINS, X_BINS, NUM_CHANNELS_XY_VEL), dtype=np.float32)
    if 'x' not in frame_points_df.columns or 'y' not in frame_points_df.columns: return heatmap
    x_coords = frame_points_df['x'].to_numpy(); y_coords = frame_points_df['y'].to_numpy()
    x_indices = np.floor((x_coords - X_MIN_METERS) / (X_MAX_METERS - X_MIN_METERS) * X_BINS).astype(int)
    y_indices = np.floor((y_coords - Y_MIN_METERS) / (Y_MAX_METERS - Y_MIN_METERS) * Y_BINS).astype(int)
    valid_indices = (x_indices >= 0) & (x_indices < X_BINS) & (y_indices >= 0) & (y_indices < Y_BINS)
    x_indices_valid = x_indices[valid_indices]; y_indices_valid = y_indices[valid_indices]
    points_in_frame_valid = frame_points_df[valid_indices]
    if points_in_frame_valid.empty: return heatmap

    if snr_feature == 'count': np.add.at(heatmap[:, :, 0], (y_indices_valid, x_indices_valid), 1)
    elif snr_feature in points_in_frame_valid.columns:
        snr_values = points_in_frame_valid[snr_feature].to_numpy()
        np.maximum.at(heatmap[:, :, 0], (y_indices_valid, x_indices_valid), snr_values)
    else: np.add.at(heatmap[:, :, 0], (y_indices_valid, x_indices_valid), 1)

    if NUM_CHANNELS_XY_VEL > 1 and velocity_feature in points_in_frame_valid.columns:
        velocity_values = points_in_frame_valid[velocity_feature].to_numpy()
        temp_vel_sum = np.zeros((Y_BINS, X_BINS), dtype=np.float32)
        temp_vel_count = np.zeros((Y_BINS, X_BINS), dtype=np.int32)
        np.add.at(temp_vel_sum, (y_indices_valid, x_indices_valid), velocity_values)
        np.add.at(temp_vel_count, (y_indices_valid, x_indices_valid), 1)
        heatmap[:, :, 1] = np.divide(temp_vel_sum, temp_vel_count, out=np.zeros_like(temp_vel_sum), where=temp_vel_count!=0)
    elif NUM_CHANNELS_XY_VEL > 1: # if velocity_feature does not exist but channel is allocated, fill with zero
        heatmap[:, :, 1] = 0
    return heatmap

# --- Preparing Heatmap Sequences, Labels, and Metadata (UPDATED) ---
def create_sequences_labels_and_metadata(df_point_cloud_data, frame_id_col,
                                          threshold_inside_tof, region_window_frames,
                                          sequence_length):
    print(f"  Starting sequence creation, labeling, and metadata collection.")
    print(f"  Sequence length: {sequence_length}, Region window: +/- {region_window_frames}")

    meta_tof_cols = ['experiment_type', frame_id_col] + \
                    [f'box{i}_distance' for i in range(1,6) if f'box{i}_distance' in df_point_cloud_data.columns] + \
                    [f'box{i}_active' for i in range(1,6) if f'box{i}_active' in df_point_cloud_data.columns]
    meta_tof_cols = list(set(meta_tof_cols))
    df_frames_meta = df_point_cloud_data[meta_tof_cols].drop_duplicates(subset=[frame_id_col]).reset_index(drop=True)
    core_tof_labels = np.zeros(len(df_frames_meta), dtype=int)
    min_distances_active = np.full(len(df_frames_meta), np.inf)
    for box_num in range(1, 6):
        active_col = f'box{box_num}_active'; dist_col = f'box{box_num}_distance'
        if active_col in df_frames_meta.columns and dist_col in df_frames_meta.columns:
            condition = (df_frames_meta[active_col] == 1) & (df_frames_meta[dist_col] < threshold_inside_tof) & (df_frames_meta[dist_col] < min_distances_active)
            core_tof_labels[condition] = box_num
            min_distances_active[condition] = df_frames_meta[dist_col][condition]
    df_frames_meta['core_tof_label'] = core_tof_labels
    refined_regional_labels = np.zeros(len(df_frames_meta), dtype=int)
    if region_window_frames >= 0:
        for exp_type, group in df_frames_meta.groupby('experiment_type', sort=False):
            exp_indices_orig = group.index.tolist()
            current_exp_core_labels = group['core_tof_label'].to_numpy()
            current_exp_regional_labels = np.zeros_like(current_exp_core_labels)
            for box_to_process in range(1, NUM_CLASSES):
                core_activation_indices_in_exp = np.where(current_exp_core_labels == box_to_process)[0]
                for core_idx in core_activation_indices_in_exp:
                    start_window_idx = max(0, core_idx - region_window_frames)
                    end_window_idx = min(len(current_exp_core_labels) - 1, core_idx + region_window_frames)
                    for target_idx_in_exp in range(start_window_idx, end_window_idx + 1):
                        if current_exp_core_labels[target_idx_in_exp] == 0 or current_exp_core_labels[target_idx_in_exp] == box_to_process:
                            current_exp_regional_labels[target_idx_in_exp] = box_to_process
            for i, original_df_idx in enumerate(exp_indices_orig):
                refined_regional_labels[original_df_idx] = current_exp_regional_labels[i]
    else: refined_regional_labels = core_tof_labels.copy()
    df_frames_meta['ground_truth_label'] = refined_regional_labels

    if not pd.api.types.is_datetime64_any_dtype(df_point_cloud_data[frame_id_col]):
        df_point_cloud_data[frame_id_col] = pd.to_datetime(df_point_cloud_data[frame_id_col], errors='coerce')
    df_point_cloud_data = df_point_cloud_data.sort_values(by=frame_id_col)
    grouped_mmwave_points = df_point_cloud_data.groupby(frame_id_col)
    
    if not pd.api.types.is_datetime64_any_dtype(df_frames_meta[frame_id_col]):
        df_frames_meta[frame_id_col] = pd.to_datetime(df_frames_meta[frame_id_col], errors='coerce')
    
    map_frame_id_to_meta_row = {row[frame_id_col]: row for index, row in df_frames_meta.iterrows()}
    temp_heatmaps_ordered = [None] * len(df_frames_meta) # heatmaps in the same order as df_frames_meta
    df_frames_meta_indices = df_frames_meta.index # original indices of df_frames_meta

    print("    Generating heatmaps per frame (SNR+Velocity channeled)...")
    processed_heatmap_count = 0
    for frame_ts, points_df in grouped_mmwave_points:
        # Check if there is a correspondence for this frame_ts in df_frames_meta
        # and place the heatmap according to its order in df_frames_meta.
        # Instead of using map_frame_id_to_meta_row, making frame_id_col of df_frames_meta
        # an index and accessing with .loc might be safer, especially if there are duplicate timestamps.
        # However, we already did drop_duplicates on df_frames_meta.
        
        # Find which row in df_frames_meta it corresponds to
        # This part might be a bit slow on very large datasets, but it can be optimized as it is sorted.
        try:
            # Get the index in df_frames_meta that corresponds to this frame_ts
            meta_row_index_in_df_frames = df_frames_meta[df_frames_meta[frame_id_col] == frame_ts].index[0]
        except IndexError:
            # This frame_ts from point_cloud_data is not in df_frames_meta (e.g., due to dropna or other filtering)
            continue

        vel_feat = 'doppler' if 'doppler' in points_df.columns else None
        heatmap_xy_vel = create_xy_heatmap_with_velocity(points_df, snr_feature='snr', velocity_feature=vel_feat)
        temp_heatmaps_ordered[meta_row_index_in_df_frames] = heatmap_xy_vel # Place at the correct index
        processed_heatmap_count +=1
        if processed_heatmap_count % 500 == 0:
             print(f"      {processed_heatmap_count}/{len(df_frames_meta)} heatmap processed.")

    all_single_frame_heatmaps = [hm if hm is not None else np.zeros((Y_BINS, X_BINS, NUM_CHANNELS_XY_VEL)) for hm in temp_heatmaps_ordered]
    all_single_frame_heatmaps_np = np.array(all_single_frame_heatmaps)
    
    print("    Creating heatmap sequences, labels, and metadata...")
    X_sequences = []
    y_sequence_labels = []
    meta_data_for_sequences = [] # Store metadata for the last frame of each sequence

    for exp_type, group_meta in df_frames_meta.groupby('experiment_type', sort=False):
        exp_original_indices = group_meta.index.tolist() # original indices within df_frames_meta
        num_frames_in_exp = len(group_meta)
        if num_frames_in_exp < sequence_length: continue

        current_exp_heatmaps = all_single_frame_heatmaps_np[exp_original_indices]
        current_exp_labels = group_meta['ground_truth_label'].to_numpy()
        # Also get metadata for this experiment
        current_exp_meta_df = group_meta.copy() # Let's take a copy

        for i in range(num_frames_in_exp - sequence_length + 1):
            sequence = current_exp_heatmaps[i : i + sequence_length]
            label_idx_in_exp = i + sequence_length - 1
            label = current_exp_labels[label_idx_in_exp]
            
            X_sequences.append(sequence)
            y_sequence_labels.append(label)
            
            # Store metadata for the last frame of the sequence
            # current_exp_meta_df is already filtered by experiment and sorted
            meta_data_for_sequences.append(current_exp_meta_df.iloc[label_idx_in_exp].to_dict())

    if not X_sequences:
        print("ERROR: No sequences could be created."); return np.array([]), np.array([]), pd.DataFrame()

    X_sequences_np = np.array(X_sequences)
    y_sequence_labels_np = np.array(y_sequence_labels)
    df_meta_sequences_pd = pd.DataFrame(meta_data_for_sequences)
    if frame_id_col in df_meta_sequences_pd.columns: # let's make frame_id_col datetime again
         df_meta_sequences_pd[frame_id_col] = pd.to_datetime(df_meta_sequences_pd[frame_id_col], errors='coerce')


    print(f"  Sequence creation complete. X_sequences: {X_sequences_np.shape}, y_labels: {y_sequence_labels_np.shape}, df_meta_sequences: {df_meta_sequences_pd.shape}")
    if y_sequence_labels_np.size > 0:
        print(f"  Sequence label distribution: {pd.Series(y_sequence_labels_np).value_counts(normalize=True).sort_index()}")
    
    return X_sequences_np, y_sequence_labels_np, df_meta_sequences_pd


# --- Plotting Functions ---
def plot_confusion_matrix_custom(y_true, y_pred, classes_labels_str, output_filename='confusion_matrix_cnn.eps', title='Confusion Matrix (CNN)'):
    unique_numeric_labels = sorted(list(set(y_true) | set(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=unique_numeric_labels)
    cm_sum = cm.sum(axis=1)[:, np.newaxis]; cm_normalized = np.divide(cm.astype('float'), cm_sum, out=np.zeros_like(cm, dtype=float), where=cm_sum!=0)
    cm_tick_labels = [classes_labels_str[l] for l in unique_numeric_labels if l < len(classes_labels_str)]
    if len(cm_tick_labels) != len(unique_numeric_labels): cm_tick_labels = [str(l) for l in unique_numeric_labels] # Fallback
    plt.figure(figsize=(max(8, len(cm_tick_labels)*1.2), max(6, len(cm_tick_labels)))); sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap=plt.cm.Blues, xticklabels=cm_tick_labels, yticklabels=cm_tick_labels, vmin=0, vmax=1)
    plt.title(title, fontsize=16); plt.ylabel('True Label', fontsize=14); plt.xlabel('Predicted Label', fontsize=14); plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0); plt.tight_layout()
    try: plt.savefig(output_filename, format='eps', dpi=300, bbox_inches='tight'); print(f"Confusion matrix: {os.path.abspath(output_filename)}")
    except Exception as e: print(f"CM saving error: {e}")
    plt.show(block=False); plt.pause(1)

def plot_per_experiment_accuracy_cnn(df_test_metadata_with_preds, target_col_true='label', target_col_pred='predicted_label_cnn', output_filename='per_exp_acc_cnn.eps'):
    if 'experiment_type' not in df_test_metadata_with_preds.columns: print("Warning: 'experiment_type' not in metadata."); return
    accuracies = {}; experiment_types = df_test_metadata_with_preds['experiment_type'].unique()
    title='Accuracy by Experiment Type'; print(f"\nAccuracy by experiment type ({title}):")
    for exp_type in experiment_types:
        exp_df_subset = df_test_metadata_with_preds[df_test_metadata_with_preds['experiment_type'] == exp_type]
        if exp_df_subset.empty or target_col_true not in exp_df_subset.columns or target_col_pred not in exp_df_subset.columns: continue
        acc = accuracy_score(exp_df_subset[target_col_true], exp_df_subset[target_col_pred]) * 100
        accuracies[exp_type] = acc; print(f"  {exp_type} ({len(exp_df_subset)} samples): {acc:.2f}%")
    if not accuracies: print("No accuracy to plot per experiment."); return
    plt.figure(figsize=(max(8, len(accuracies)*1.5), 6)); exp_names = list(accuracies.keys()); acc_values = list(accuracies.values())
    bars = plt.bar(exp_names, acc_values, color=sns.color_palette("viridis", len(exp_names))); plt.title(title, fontsize=16); plt.xlabel('Experiment Type', fontsize=14); plt.ylabel('Accuracy (%)', fontsize=14); plt.xticks(rotation=45, ha='right'); plt.ylim(0, 105)
    for bar in bars: yval = bar.get_height(); plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.1f}%', ha='center', va='bottom', fontsize=10)
    plt.tight_layout();
    try: plt.savefig(output_filename, format='eps', dpi=300, bbox_inches='tight'); print(f"Per-experiment accuracy plot: {os.path.abspath(output_filename)}")
    except Exception as e: print(f"Plot saving error: {e}")
    plt.show(block=False); plt.pause(1)

def visualize_boxes_over_time_cnn(df_vis_data, frame_id_col_name, true_col, pred_col, accuracy_val, output_filename='boxes_time_cnn.eps', max_points=1000):
    df_vis_data_copy = df_vis_data.copy()
    time_col_for_plot = frame_id_col_name

    if time_col_for_plot not in df_vis_data_copy.columns or df_vis_data_copy[time_col_for_plot].isnull().all():
        print(f"Warning: Time column '{time_col_for_plot}' not found/NaN. Using index.")
        df_vis_data_copy = df_vis_data_copy.reset_index() # Add index as a column
        time_col_for_plot = 'index' # New time column became 'index'
        x_label = 'Sample Index (End of Sequence)'
    else:
        x_label = f'{time_col_for_plot} (End of Sequence)'
        if pd.api.types.is_datetime64_any_dtype(df_vis_data_copy[time_col_for_plot]):
            df_vis_data_copy = df_vis_data_copy.sort_values(by=time_col_for_plot)
        elif pd.api.types.is_numeric_dtype(df_vis_data_copy[time_col_for_plot]):
            df_vis_data_copy = df_vis_data_copy.sort_values(by=time_col_for_plot)
        else: # If not datetime or numeric, it might be string or object, sorting might be meaningless
             print(f"Warning: Time column '{time_col_for_plot}' is not a sortable type. Plotting order will be according to DataFrame order.")


    df_vis_data_copy[true_col] = pd.to_numeric(df_vis_data_copy[true_col], errors='coerce')
    df_vis_data_copy[pred_col] = pd.to_numeric(df_vis_data_copy[pred_col], errors='coerce')
    # if time_col_for_plot is NaN, dropna won't work, but it's fine if it's 'index'.
    df_vis_data_copy.dropna(subset=[true_col, pred_col] + ([time_col_for_plot] if time_col_for_plot != 'index' else []), inplace=True)

    if df_vis_data_copy.empty: print("No data left for visualization (after NaN)."); return

    num_rows = len(df_vis_data_copy)
    if num_rows > max_points:
        indices_to_sample = np.linspace(0, num_rows - 1, max_points, dtype=int)
        df_sample = df_vis_data_copy.iloc[indices_to_sample].copy()
    else:
        df_sample = df_vis_data_copy.copy()

    plt.figure(figsize=(15, 7))
    plt.plot(df_sample[time_col_for_plot].to_numpy(), df_sample[true_col].to_numpy(), 'o-', markersize=4, label='True Box (Refined Regional)', color='royalblue', alpha=0.8)
    plt.plot(df_sample[time_col_for_plot].to_numpy(), df_sample[pred_col].to_numpy(), 'x--', markersize=5, label='Predicted Box (CNN-LSTM)', color='orangered', alpha=0.8)
    
    all_box_values_true = df_sample[true_col].dropna().astype(int).unique()
    all_box_values_pred = df_sample[pred_col].dropna().astype(int).unique()
    all_box_values = set(all_box_values_true) | set(all_box_values_pred)
    
    min_b, max_b = (min(all_box_values), max(all_box_values)) if all_box_values else (0, NUM_CLASSES - 1)
    
    labels_map = {i: f'Box {i}' if i!=0 else 'No Box' for i in range(0, int(max_b) + 2)}
    y_ticks_num = sorted(list(all_box_values))
    y_tick_lab = [labels_map.get(b, f'Unknown {b}') for b in y_ticks_num]
    
    if y_ticks_num: plt.yticks(ticks=y_ticks_num, labels=y_tick_lab, fontsize=12)
        
    plt.title(f'Hand Position Prediction (CNN-LSTM) (Accuracy: {accuracy_val:.2f}%)', fontsize=16)
    plt.xlabel(x_label, fontsize=14); plt.ylabel('Box Number', fontsize=14); plt.legend(fontsize=12); plt.grid(True, alpha=0.7); plt.tight_layout()
    try: plt.savefig(output_filename, format='eps', dpi=300, bbox_inches='tight'); print(f"Time series plot: {os.path.abspath(output_filename)}")
    except Exception as e: print(f"Visualization saving error: {e}")
    plt.show(block=False); plt.pause(1); return plt


# --- Main Script ---
print(f"--- CNN-LSTM (X-Y Heatmap + Velocity, Refined Regional GT) Training ---")
print(f"Sequence Length: {SEQUENCE_LENGTH}, Number of Heatmap Channels: {NUM_CHANNELS_XY_VEL}")

# 1. Load Data
data_paths = discover_data_files(BASE_DATA_DIRECTORY)
if not data_paths: raise SystemExit("Data file not found.")
df_point_cloud_with_tof_loaded = load_and_preprocess_data(data_paths, FRAME_ID_COLUMN)
if df_point_cloud_with_tof_loaded.empty: raise SystemExit("Loaded data is empty.")

# 2. Prepare Heatmap Sequences, Labels, and Metadata for ALL DATA
print("\n--- Preparing Heatmap Sequences, Labels, and Metadata for All Data ---")
X_sequences_all, y_labels_all_refined, df_meta_all_sequences = create_sequences_labels_and_metadata(
    df_point_cloud_with_tof_loaded, FRAME_ID_COLUMN,
    threshold_inside_tof=THRESHOLD_INSIDE_TOF,
    region_window_frames=REGION_WINDOW_SIZE_FRAMES,
    sequence_length=SEQUENCE_LENGTH
)

if X_sequences_all.size == 0 or y_labels_all_refined.size == 0:
    raise SystemExit("Sequence or labels could not be created.")

# 3. Data Splitting (Train/Test) - with metadata
print("\n--- Data Splitting (Train/Test) ---")
test_set_fraction = 0.2; random_state_split = 42
stratify_opt_seq = y_labels_all_refined if len(np.unique(y_labels_all_refined)) > 1 else None

X_train_seq, X_test_seq, \
y_train_labels_refined, y_test_labels_refined, \
df_meta_train_seq, df_meta_test_seq = train_test_split(
    X_sequences_all, y_labels_all_refined, df_meta_all_sequences,
    test_size=test_set_fraction, random_state=random_state_split, stratify=stratify_opt_seq
)
print(f"X_train (sequences): {X_train_seq.shape}, y_train (GT refined): {y_train_labels_refined.shape}, df_meta_train_seq: {df_meta_train_seq.shape}")
print(f"X_test (sequences): {X_test_seq.shape}, y_test (GT refined): {y_test_labels_refined.shape}, df_meta_test_seq: {df_meta_test_seq.shape}")
print(f"Training set GT (refined) label distribution: {pd.Series(y_train_labels_refined).value_counts(normalize=True).sort_index()}")
print(f"Test set GT (refined) label distribution: {pd.Series(y_test_labels_refined).value_counts(normalize=True).sort_index()}")


# 4. Creating CNN-LSTM Model
print("\n--- Creating CNN-LSTM Model ---")
input_shape_seq = (SEQUENCE_LENGTH, IMG_HEIGHT_XY, IMG_WIDTH_XY, NUM_CHANNELS_XY_VEL)
cnn_input_shape_single_frame = (IMG_HEIGHT_XY, IMG_WIDTH_XY, NUM_CHANNELS_XY_VEL)

cnn_base = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=cnn_input_shape_single_frame), BatchNormalization(), MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D((2,2)),
    Conv2D(128, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D((2,2)), Flatten()
], name="cnn_base")
video_input = Input(shape=input_shape_seq, name="video_input")
encoded_frames = TimeDistributed(cnn_base, name="time_distributed_cnn")(video_input)
encoded_sequence = GRU(128, dropout=0.3, recurrent_dropout=0.2, name="gru_layer")(encoded_frames) # GRU used
output_layer = Dense(NUM_CLASSES, activation='softmax', name="output_layer")(encoded_sequence)
model_cnnlstm = Model(inputs=video_input, outputs=output_layer, name="cnn_lstm_model")

lr_sched_cnnlstm = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=0.0005, decay_steps=8000, decay_rate=0.92)
model_cnnlstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_sched_cnnlstm),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_cnnlstm.summary()


# 5. Model Training
print("\n--- CNN-LSTM Model Training ---")
epochs_cnnlstm = 120; batch_cnnlstm = 32 # Epochs can be increased, rely on patience
unique_tr_labels_cnnlstm = np.unique(y_train_labels_refined)
class_weights_cnnlstm = None
if len(unique_tr_labels_cnnlstm) > 1:
    class_w_calc_cnnlstm = compute_class_weight('balanced', classes=unique_tr_labels_cnnlstm, y=y_train_labels_refined)
    class_weights_cnnlstm = dict(zip(unique_tr_labels_cnnlstm, class_w_calc_cnnlstm)); print(f"Class weights: {class_weights_cnnlstm}")

callbacks_cnnlstm = [
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=1),
    ModelCheckpoint('../models/best_cnnlstm_model_vel_v2.keras', monitor='val_accuracy', save_best_only=True, verbose=1, mode='max')
]
history_cnnlstm = model_cnnlstm.fit(
    X_train_seq, y_train_labels_refined,
    epochs=epochs_cnnlstm, batch_size=batch_cnnlstm,
    validation_data=(X_test_seq, y_test_labels_refined),
    callbacks=callbacks_cnnlstm, class_weight=class_weights_cnnlstm, verbose=1
)
best_model_cnnlstm_file = '../models/best_cnnlstm_model_vel_v2.keras'
if os.path.exists(best_model_cnnlstm_file):
    print(f"Loading best CNN-LSTM model '{best_model_cnnlstm_file}'...")
    model_cnnlstm = tf.keras.models.load_model(best_model_cnnlstm_file)

# 6. Model Evaluation
print("\n--- CNN-LSTM Model Evaluation (TEST SET - REFINED GT) ---")
loss_test_cnnlstm, acc_test_cnnlstm = model_cnnlstm.evaluate(X_test_seq, y_test_labels_refined, verbose=0)
acc_test_cnnlstm *= 100
print(f"CNN-LSTM Test Loss (Refined GT): {loss_test_cnnlstm:.4f}")
print(f"CNN-LSTM Test Accuracy (Refined GT): {acc_test_cnnlstm:.2f}%")

y_pred_proba_cnnlstm = model_cnnlstm.predict(X_test_seq)
y_pred_classes_cnnlstm = np.argmax(y_pred_proba_cnnlstm, axis=1)
class_labels_str_all = [f'Box {l}' if l != 0 else 'No Box' for l in range(NUM_CLASSES)]
present_labels_num_cnnlstm = sorted(list(set(y_test_labels_refined) | set(y_pred_classes_cnnlstm)))
report_target_names_cnnlstm = [class_labels_str_all[l] for l in present_labels_num_cnnlstm if l < NUM_CLASSES]

print("\nClassification Report (CNN-LSTM Test Set - Refined GT):")
try:
    class_report_cnnlstm = classification_report(
        y_test_labels_refined, y_pred_classes_cnnlstm,
        labels=present_labels_num_cnnlstm, target_names=report_target_names_cnnlstm, zero_division=0
    )
    print(class_report_cnnlstm)
    report_path_cnnlstm = 'classification_report_CNN_LSTM_vel_v2.txt'
    with open(report_path_cnnlstm, 'w') as f:
        f.write(f"CNN-LSTM Test Accuracy (Refined GT): {acc_test_cnnlstm:.2f}%\n\n{class_report_cnnlstm}")
    print(f"Report saved: {os.path.abspath(report_path_cnnlstm)}")
except Exception as e: print(f"Classification report error: {e}")

# 7. Plots (Will now work with df_meta_test_seq)
print("\n--- CNN-LSTM Plots ---")
plot_confusion_matrix_custom(y_test_labels_refined, y_pred_classes_cnnlstm, class_labels_str_all,
                             output_filename='confusion_matrix_CNN_LSTM_vel_v2.eps',
                             title='Confusion Matrix (CNN-LSTM - Refined GT)')

# Add predictions to df_meta_test_seq
df_meta_test_seq_copy = df_meta_test_seq.copy() # Plotting functions might modify the DataFrame.
if len(df_meta_test_seq_copy) == len(y_pred_classes_cnnlstm): # Size check
    df_meta_test_seq_copy['predicted_label_cnnlstm'] = y_pred_classes_cnnlstm
    # True labels should already be in df_meta_test_seq as 'ground_truth_label'
    # (check the 'ground_truth_label' column of the df coming from create_sequences_labels_and_metadata)
    # If 'ground_truth_label' does not exist, use y_test_labels_refined.
    if 'ground_truth_label' not in df_meta_test_seq_copy.columns:
         df_meta_test_seq_copy['ground_truth_label'] = y_test_labels_refined

    plot_per_experiment_accuracy_cnn(df_meta_test_seq_copy,
                                     target_col_true='ground_truth_label', # column name in df_meta_test_seq
                                     target_col_pred='predicted_label_cnnlstm',
                                     output_filename='per_exp_acc_CNN_LSTM_vel_v2.eps')

    visualize_boxes_over_time_cnn(df_meta_test_seq_copy, FRAME_ID_COLUMN, # FRAME_ID_COLUMN should be in df_meta_test_seq
                                  'ground_truth_label', 'predicted_label_cnnlstm', acc_test_cnnlstm,
                                  output_filename='boxes_time_CNN_LSTM_vel_v2.eps')
else:
    print(f"WARNING: Size mismatch between df_meta_test_seq ({len(df_meta_test_seq_copy)}) and CNN-LSTM predictions ({len(y_pred_classes_cnnlstm)}). Skipping time series plots.")


# 8. Saving Model and Configuration
model_dir_cnnlstm = 'saved_model_cnnlstm_vel_v2'
os.makedirs(model_dir_cnnlstm, exist_ok=True)
config_path_cnnlstm = os.path.join(model_dir_cnnlstm, 'config_cnnlstm_vel_v2.txt')
with open(config_path_cnnlstm, 'w') as f:
    f.write(f"MODEL_TYPE=CNN_LSTM_with_Velocity_v2\n")
    f.write(f"BASE_DATA_DIRECTORY={BASE_DATA_DIRECTORY}\nFRAME_ID_COLUMN={FRAME_ID_COLUMN}\nNUM_CLASSES={NUM_CLASSES}\n")
    f.write(f"THRESHOLD_INSIDE_TOF={THRESHOLD_INSIDE_TOF}\nREGION_WINDOW_SIZE_FRAMES={REGION_WINDOW_SIZE_FRAMES}\n")
    f.write(f"SEQUENCE_LENGTH={SEQUENCE_LENGTH}\n")
    f.write(f"HEATMAP_TYPE=X-Y_TopView_SNR_and_Velocity\nX_MIN_METERS={X_MIN_METERS}, X_MAX_METERS={X_MAX_METERS}\n")
    f.write(f"Y_MIN_METERS={Y_MIN_METERS}, Y_MAX_METERS={Y_MAX_METERS}\n")
    f.write(f"X_BINS={X_BINS}, Y_BINS={Y_BINS}, NUM_CHANNELS_XY_VEL={NUM_CHANNELS_XY_VEL}\n")
    f.write(f"Test Set Fraction: {test_set_fraction}\nEpochs: {epochs_cnnlstm}, Batch Size: {batch_cnnlstm}\n")
    f.write(f"Final Test Accuracy (Refined GT): {acc_test_cnnlstm:.2f}%\n")
print(f"Configuration saved: {config_path_cnnlstm}")
print(f"Best CNN-LSTM model saved as '{os.path.abspath(best_model_cnnlstm_file)}'.")
plt.show(block=True) # Show all plots at the end of the script and keep them open
print("\n--- CNN-LSTM (X-Y Heatmap + Velocity, Refined Regional GT) Training Script Completed ---")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import os
import glob

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, LSTM, TimeDistributed, GRU
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
# from tensorflow.keras.preprocessing.sequence import pad_sequences # Not used for now

# --- Configuration ---
BASE_DATA_DIRECTORY = '../data' # UPDATE YOUR DATA PATH
FRAME_ID_COLUMN = 'timestamp_mmwave'
NUM_CLASSES = 6

THRESHOLD_INSIDE_TOF = 140
REGION_WINDOW_SIZE_FRAMES = 3

# mmWave X-Y Heatmap Configuration
X_MIN_METERS = -0.5; X_MAX_METERS = 0.5
Y_MIN_METERS = 0.1; Y_MAX_METERS = 1.0
X_BINS = 48; Y_BINS = 48
NUM_CHANNELS_XY_VEL = 2 # Channel 0: SNR/Count, Channel 1: Velocity

IMG_HEIGHT_XY = Y_BINS
IMG_WIDTH_XY = X_BINS

# Time Series Configuration
SEQUENCE_LENGTH = 5

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

# --- Data Discovery and Loading Functions ---
def discover_data_files(base_dir):
    discovered_paths_map = {}
    base_dir = os.path.normpath(base_dir)
    if not os.path.isdir(base_dir): print(f"Error: Base data directory '{base_dir}' not found."); return discovered_paths_map
    search_pattern = os.path.join(base_dir, '**', 'merged_df.csv')
    found_files = sorted(list(glob.glob(search_pattern, recursive=True)))
    if not found_files: print(f"Warning: No files found with '{search_pattern}' under '{base_dir}'."); return discovered_paths_map
    file_counter = 0
    for filepath in found_files:
        try:
            relative_path = os.path.relpath(filepath, base_dir)
            path_parts = relative_path.split(os.sep)
            experiment_type = path_parts[-2] if len(path_parts) > 1 and path_parts[-1] == 'merged_df.csv' else f"untyped_experiment_{file_counter}"
            if experiment_type == '.': experiment_type = f"untyped_experiment_{file_counter}"
            file_counter +=1; normalized_filepath = os.path.normpath(filepath)
            if experiment_type not in discovered_paths_map: discovered_paths_map[experiment_type] = []
            discovered_paths_map[experiment_type].append(normalized_filepath)
        except Exception as e: print(f"  Error while processing '{filepath}': {e}")
    if not discovered_paths_map: print(f"Warning: Data file could not be mapped from directory '{base_dir}'.")
    else: print(f"\nSuccessfully discovered data files: {len(discovered_paths_map)} experiment type(s).")
    return discovered_paths_map

def load_and_preprocess_data(data_paths_map_with_lists, frame_id_col_name):
    all_dataframes_merged = []
    for experiment_type, path_list in data_paths_map_with_lists.items():
        dataframes_for_current_type = []
        for path in path_list:
            if os.path.exists(path):
                try:
                    df = pd.read_csv(path); df['experiment_type'] = experiment_type
                    dataframes_for_current_type.append(df)
                except Exception as e: print(f"  Error while loading '{path}': {e}")
        if dataframes_for_current_type: all_dataframes_merged.append(pd.concat(dataframes_for_current_type, ignore_index=True))
    if not all_dataframes_merged: raise FileNotFoundError("No valid data file was loaded.")
    final_merged_df = pd.concat(all_dataframes_merged, ignore_index=True)
    print(f"Final merged (point level) data shape: {final_merged_df.shape}")
    datetime_cols_to_convert = list(set(['timestamp_mmwave', 'timestamp_dist', 'timestamp', frame_id_col_name]))
    for col in datetime_cols_to_convert:
        if col in final_merged_df.columns:
            if not pd.api.types.is_datetime64_any_dtype(final_merged_df[col]):
                try:
                    if pd.api.types.is_numeric_dtype(final_merged_df[col]): final_merged_df[col] = pd.to_datetime(final_merged_df[col], unit='s', errors='coerce')
                    else: final_merged_df[col] = pd.to_datetime(final_merged_df[col], errors='coerce')
                except Exception as e: print(f"Warning: '{col}' could not be converted to datetime: {e}.")
    if frame_id_col_name in final_merged_df.columns and pd.api.types.is_datetime64_any_dtype(final_merged_df[frame_id_col_name]):
        final_merged_df = final_merged_df.sort_values(by=[frame_id_col_name, 'experiment_type']).reset_index(drop=True)
    elif frame_id_col_name in final_merged_df.columns:
        final_merged_df = final_merged_df.sort_values(by=[frame_id_col_name, 'experiment_type']).reset_index(drop=True)
        print(f"Warning: '{frame_id_col_name}' is not datetime, sorted numerically.")
    else: print(f"WARNING: '{frame_id_col_name}' cannot be used for sorting.")
    return final_merged_df

# --- mmWave X-Y Heatmap Generation (VELOCITY CHANNEL ADDED) ---
def create_xy_heatmap_with_velocity(frame_points_df, snr_feature='snr', velocity_feature='doppler'):
    heatmap = np.zeros((Y_BINS, X_BINS, NUM_CHANNELS_XY_VEL), dtype=np.float32)
    if 'x' not in frame_points_df.columns or 'y' not in frame_points_df.columns: return heatmap
    x_coords = frame_points_df['x'].to_numpy(); y_coords = frame_points_df['y'].to_numpy()
    x_indices = np.floor((x_coords - X_MIN_METERS) / (X_MAX_METERS - X_MIN_METERS) * X_BINS).astype(int)
    y_indices = np.floor((y_coords - Y_MIN_METERS) / (Y_MAX_METERS - Y_MIN_METERS) * Y_BINS).astype(int)
    valid_indices = (x_indices >= 0) & (x_indices < X_BINS) & (y_indices >= 0) & (y_indices < Y_BINS)
    x_indices_valid = x_indices[valid_indices]; y_indices_valid = y_indices[valid_indices]
    points_in_frame_valid = frame_points_df[valid_indices]
    if points_in_frame_valid.empty: return heatmap

    if snr_feature == 'count': np.add.at(heatmap[:, :, 0], (y_indices_valid, x_indices_valid), 1)
    elif snr_feature in points_in_frame_valid.columns:
        snr_values = points_in_frame_valid[snr_feature].to_numpy()
        np.maximum.at(heatmap[:, :, 0], (y_indices_valid, x_indices_valid), snr_values)
    else: np.add.at(heatmap[:, :, 0], (y_indices_valid, x_indices_valid), 1)

    if NUM_CHANNELS_XY_VEL > 1 and velocity_feature in points_in_frame_valid.columns:
        velocity_values = points_in_frame_valid[velocity_feature].to_numpy()
        temp_vel_sum = np.zeros((Y_BINS, X_BINS), dtype=np.float32)
        temp_vel_count = np.zeros((Y_BINS, X_BINS), dtype=np.int32)
        np.add.at(temp_vel_sum, (y_indices_valid, x_indices_valid), velocity_values)
        np.add.at(temp_vel_count, (y_indices_valid, x_indices_valid), 1)
        heatmap[:, :, 1] = np.divide(temp_vel_sum, temp_vel_count, out=np.zeros_like(temp_vel_sum), where=temp_vel_count!=0)
    elif NUM_CHANNELS_XY_VEL > 1: # if velocity_feature does not exist but channel is allocated, fill with zero
        heatmap[:, :, 1] = 0
    return heatmap

# --- Preparing Heatmap Sequences, Labels, and Metadata (UPDATED) ---
def create_sequences_labels_and_metadata(df_point_cloud_data, frame_id_col,
                                          threshold_inside_tof, region_window_frames,
                                          sequence_length):
    print(f"  Starting sequence creation, labeling, and metadata collection.")
    print(f"  Sequence length: {sequence_length}, Region window: +/- {region_window_frames}")

    meta_tof_cols = ['experiment_type', frame_id_col] + \
                    [f'box{i}_distance' for i in range(1,6) if f'box{i}_distance' in df_point_cloud_data.columns] + \
                    [f'box{i}_active' for i in range(1,6) if f'box{i}_active' in df_point_cloud_data.columns]
    meta_tof_cols = list(set(meta_tof_cols))
    df_frames_meta = df_point_cloud_data[meta_tof_cols].drop_duplicates(subset=[frame_id_col]).reset_index(drop=True)
    core_tof_labels = np.zeros(len(df_frames_meta), dtype=int)
    min_distances_active = np.full(len(df_frames_meta), np.inf)
    for box_num in range(1, 6):
        active_col = f'box{box_num}_active'; dist_col = f'box{box_num}_distance'
        if active_col in df_frames_meta.columns and dist_col in df_frames_meta.columns:
            condition = (df_frames_meta[active_col] == 1) & (df_frames_meta[dist_col] < threshold_inside_tof) & (df_frames_meta[dist_col] < min_distances_active)
            core_tof_labels[condition] = box_num
            min_distances_active[condition] = df_frames_meta[dist_col][condition]
    df_frames_meta['core_tof_label'] = core_tof_labels
    refined_regional_labels = np.zeros(len(df_frames_meta), dtype=int)
    if region_window_frames >= 0:
        for exp_type, group in df_frames_meta.groupby('experiment_type', sort=False):
            exp_indices_orig = group.index.tolist()
            current_exp_core_labels = group['core_tof_label'].to_numpy()
            current_exp_regional_labels = np.zeros_like(current_exp_core_labels)
            for box_to_process in range(1, NUM_CLASSES):
                core_activation_indices_in_exp = np.where(current_exp_core_labels == box_to_process)[0]
                for core_idx in core_activation_indices_in_exp:
                    start_window_idx = max(0, core_idx - region_window_frames)
                    end_window_idx = min(len(current_exp_core_labels) - 1, core_idx + region_window_frames)
                    for target_idx_in_exp in range(start_window_idx, end_window_idx + 1):
                        if current_exp_core_labels[target_idx_in_exp] == 0 or current_exp_core_labels[target_idx_in_exp] == box_to_process:
                            current_exp_regional_labels[target_idx_in_exp] = box_to_process
            for i, original_df_idx in enumerate(exp_indices_orig):
                refined_regional_labels[original_df_idx] = current_exp_regional_labels[i]
    else: refined_regional_labels = core_tof_labels.copy()
    df_frames_meta['ground_truth_label'] = refined_regional_labels

    if not pd.api.types.is_datetime64_any_dtype(df_point_cloud_data[frame_id_col]):
        df_point_cloud_data[frame_id_col] = pd.to_datetime(df_point_cloud_data[frame_id_col], errors='coerce')
    df_point_cloud_data = df_point_cloud_data.sort_values(by=frame_id_col)
    grouped_mmwave_points = df_point_cloud_data.groupby(frame_id_col)
    
    if not pd.api.types.is_datetime64_any_dtype(df_frames_meta[frame_id_col]):
        df_frames_meta[frame_id_col] = pd.to_datetime(df_frames_meta[frame_id_col], errors='coerce')
    
    map_frame_id_to_meta_row = {row[frame_id_col]: row for index, row in df_frames_meta.iterrows()}
    temp_heatmaps_ordered = [None] * len(df_frames_meta) # heatmaps in the same order as df_frames_meta
    df_frames_meta_indices = df_frames_meta.index # original indices of df_frames_meta

    print("    Generating heatmaps per frame (SNR+Velocity channeled)...")
    processed_heatmap_count = 0
    for frame_ts, points_df in grouped_mmwave_points:
        # Check if there is a correspondence for this frame_ts in df_frames_meta
        # and place the heatmap according to its order in df_frames_meta.
        # Instead of using map_frame_id_to_meta_row, making frame_id_col of df_frames_meta
        # an index and accessing with .loc might be safer, especially if there are duplicate timestamps.
        # However, we already did drop_duplicates on df_frames_meta.
        
        # Find which row in df_frames_meta it corresponds to
        # This part might be a bit slow on very large datasets, but it can be optimized as it is sorted.
        try:
            # Get the index in df_frames_meta that corresponds to this frame_ts
            meta_row_index_in_df_frames = df_frames_meta[df_frames_meta[frame_id_col] == frame_ts].index[0]
        except IndexError:
            # This frame_ts from point_cloud_data is not in df_frames_meta (e.g., due to dropna or other filtering)
            continue

        vel_feat = 'doppler' if 'doppler' in points_df.columns else None
        heatmap_xy_vel = create_xy_heatmap_with_velocity(points_df, snr_feature='snr', velocity_feature=vel_feat)
        temp_heatmaps_ordered[meta_row_index_in_df_frames] = heatmap_xy_vel # Place at the correct index
        processed_heatmap_count +=1
        if processed_heatmap_count % 500 == 0:
             print(f"      {processed_heatmap_count}/{len(df_frames_meta)} heatmap processed.")

    all_single_frame_heatmaps = [hm if hm is not None else np.zeros((Y_BINS, X_BINS, NUM_CHANNELS_XY_VEL)) for hm in temp_heatmaps_ordered]
    all_single_frame_heatmaps_np = np.array(all_single_frame_heatmaps)
    
    print("    Creating heatmap sequences, labels, and metadata...")
    X_sequences = []
    y_sequence_labels = []
    meta_data_for_sequences = [] # Store metadata for the last frame of each sequence

    for exp_type, group_meta in df_frames_meta.groupby('experiment_type', sort=False):
        exp_original_indices = group_meta.index.tolist() # original indices within df_frames_meta
        num_frames_in_exp = len(group_meta)
        if num_frames_in_exp < sequence_length: continue

        current_exp_heatmaps = all_single_frame_heatmaps_np[exp_original_indices]
        current_exp_labels = group_meta['ground_truth_label'].to_numpy()
        # Also get metadata for this experiment
        current_exp_meta_df = group_meta.copy() # Let's take a copy

        for i in range(num_frames_in_exp - sequence_length + 1):
            sequence = current_exp_heatmaps[i : i + sequence_length]
            label_idx_in_exp = i + sequence_length - 1
            label = current_exp_labels[label_idx_in_exp]
            
            X_sequences.append(sequence)
            y_sequence_labels.append(label)
            
            # Store metadata for the last frame of the sequence
            # current_exp_meta_df is already filtered by experiment and sorted
            meta_data_for_sequences.append(current_exp_meta_df.iloc[label_idx_in_exp].to_dict())

    if not X_sequences:
        print("ERROR: No sequences could be created."); return np.array([]), np.array([]), pd.DataFrame()

    X_sequences_np = np.array(X_sequences)
    y_sequence_labels_np = np.array(y_sequence_labels)
    df_meta_sequences_pd = pd.DataFrame(meta_data_for_sequences)
    if frame_id_col in df_meta_sequences_pd.columns: # let's make frame_id_col datetime again
         df_meta_sequences_pd[frame_id_col] = pd.to_datetime(df_meta_sequences_pd[frame_id_col], errors='coerce')


    print(f"  Sequence creation complete. X_sequences: {X_sequences_np.shape}, y_labels: {y_sequence_labels_np.shape}, df_meta_sequences: {df_meta_sequences_pd.shape}")
    if y_sequence_labels_np.size > 0:
        print(f"  Sequence label distribution: {pd.Series(y_sequence_labels_np).value_counts(normalize=True).sort_index()}")
    
    return X_sequences_np, y_sequence_labels_np, df_meta_sequences_pd


# --- Plotting Functions ---
def plot_confusion_matrix_custom(y_true, y_pred, classes_labels_str, output_filename='confusion_matrix_cnn.eps', title='Confusion Matrix (CNN)'):
    unique_numeric_labels = sorted(list(set(y_true) | set(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=unique_numeric_labels)
    cm_sum = cm.sum(axis=1)[:, np.newaxis]; cm_normalized = np.divide(cm.astype('float'), cm_sum, out=np.zeros_like(cm, dtype=float), where=cm_sum!=0)
    cm_tick_labels = [classes_labels_str[l] for l in unique_numeric_labels if l < len(classes_labels_str)]
    if len(cm_tick_labels) != len(unique_numeric_labels): cm_tick_labels = [str(l) for l in unique_numeric_labels] # Fallback
    plt.figure(figsize=(max(8, len(cm_tick_labels)*1.2), max(6, len(cm_tick_labels)))); sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap=plt.cm.Blues, xticklabels=cm_tick_labels, yticklabels=cm_tick_labels, vmin=0, vmax=1)
    plt.title(title, fontsize=16); plt.ylabel('True Label', fontsize=14); plt.xlabel('Predicted Label', fontsize=14); plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0); plt.tight_layout()
    try: plt.savefig(output_filename, format='eps', dpi=300, bbox_inches='tight'); print(f"Confusion matrix: {os.path.abspath(output_filename)}")
    except Exception as e: print(f"CM saving error: {e}")
    plt.show(block=False); plt.pause(1)

def plot_per_experiment_accuracy_cnn(df_test_metadata_with_preds, target_col_true='label', target_col_pred='predicted_label_cnn', output_filename='per_exp_acc_cnn.eps'):
    if 'experiment_type' not in df_test_metadata_with_preds.columns: print("Warning: 'experiment_type' not in metadata."); return
    accuracies = {}; experiment_types = df_test_metadata_with_preds['experiment_type'].unique()
    title='Accuracy by Experiment Type'; print(f"\nAccuracy by experiment type ({title}):")
    for exp_type in experiment_types:
        exp_df_subset = df_test_metadata_with_preds[df_test_metadata_with_preds['experiment_type'] == exp_type]
        if exp_df_subset.empty or target_col_true not in exp_df_subset.columns or target_col_pred not in exp_df_subset.columns: continue
        acc = accuracy_score(exp_df_subset[target_col_true], exp_df_subset[target_col_pred]) * 100
        accuracies[exp_type] = acc; print(f"  {exp_type} ({len(exp_df_subset)} samples): {acc:.2f}%")
    if not accuracies: print("No accuracy to plot per experiment."); return
    plt.figure(figsize=(max(8, len(accuracies)*1.5), 6)); exp_names = list(accuracies.keys()); acc_values = list(accuracies.values())
    bars = plt.bar(exp_names, acc_values, color=sns.color_palette("viridis", len(exp_names))); plt.title(title, fontsize=16); plt.xlabel('Experiment Type', fontsize=14); plt.ylabel('Accuracy (%)', fontsize=14); plt.xticks(rotation=45, ha='right'); plt.ylim(0, 105)
    for bar in bars: yval = bar.get_height(); plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.1f}%', ha='center', va='bottom', fontsize=10)
    plt.tight_layout();
    try: plt.savefig(output_filename, format='eps', dpi=300, bbox_inches='tight'); print(f"Per-experiment accuracy plot: {os.path.abspath(output_filename)}")
    except Exception as e: print(f"Plot saving error: {e}")
    plt.show(block=False); plt.pause(1)

def visualize_boxes_over_time_cnn(df_vis_data, frame_id_col_name, true_col, pred_col, accuracy_val, output_filename='boxes_time_cnn.eps', max_points=1000):
    df_vis_data_copy = df_vis_data.copy()
    time_col_for_plot = frame_id_col_name

    if time_col_for_plot not in df_vis_data_copy.columns or df_vis_data_copy[time_col_for_plot].isnull().all():
        print(f"Warning: Time column '{time_col_for_plot}' not found/NaN. Using index.")
        df_vis_data_copy = df_vis_data_copy.reset_index() # Add index as a column
        time_col_for_plot = 'index' # New time column became 'index'
        x_label = 'Sample Index (End of Sequence)'
    else:
        x_label = f'{time_col_for_plot} (End of Sequence)'
        if pd.api.types.is_datetime64_any_dtype(df_vis_data_copy[time_col_for_plot]):
            df_vis_data_copy = df_vis_data_copy.sort_values(by=time_col_for_plot)
        elif pd.api.types.is_numeric_dtype(df_vis_data_copy[time_col_for_plot]):
            df_vis_data_copy = df_vis_data_copy.sort_values(by=time_col_for_plot)
        else: # If not datetime or numeric, it might be string or object, sorting might be meaningless
             print(f"Warning: Time column '{time_col_for_plot}' is not a sortable type. Plotting order will be according to DataFrame order.")


    df_vis_data_copy[true_col] = pd.to_numeric(df_vis_data_copy[true_col], errors='coerce')
    df_vis_data_copy[pred_col] = pd.to_numeric(df_vis_data_copy[pred_col], errors='coerce')
    # if time_col_for_plot is NaN, dropna won't work, but it's fine if it's 'index'.
    df_vis_data_copy.dropna(subset=[true_col, pred_col] + ([time_col_for_plot] if time_col_for_plot != 'index' else []), inplace=True)

    if df_vis_data_copy.empty: print("No data left for visualization (after NaN)."); return

    num_rows = len(df_vis_data_copy)
    if num_rows > max_points:
        indices_to_sample = np.linspace(0, num_rows - 1, max_points, dtype=int)
        df_sample = df_vis_data_copy.iloc[indices_to_sample].copy()
    else:
        df_sample = df_vis_data_copy.copy()

    plt.figure(figsize=(15, 7))
    plt.plot(df_sample[time_col_for_plot].to_numpy(), df_sample[true_col].to_numpy(), 'o-', markersize=4, label='True Box (Refined Regional)', color='royalblue', alpha=0.8)
    plt.plot(df_sample[time_col_for_plot].to_numpy(), df_sample[pred_col].to_numpy(), 'x--', markersize=5, label='Predicted Box (CNN-LSTM)', color='orangered', alpha=0.8)
    
    all_box_values_true = df_sample[true_col].dropna().astype(int).unique()
    all_box_values_pred = df_sample[pred_col].dropna().astype(int).unique()
    all_box_values = set(all_box_values_true) | set(all_box_values_pred)
    
    min_b, max_b = (min(all_box_values), max(all_box_values)) if all_box_values else (0, NUM_CLASSES - 1)
    
    labels_map = {i: f'Box {i}' if i!=0 else 'No Box' for i in range(0, int(max_b) + 2)}
    y_ticks_num = sorted(list(all_box_values))
    y_tick_lab = [labels_map.get(b, f'Unknown {b}') for b in y_ticks_num]
    
    if y_ticks_num: plt.yticks(ticks=y_ticks_num, labels=y_tick_lab, fontsize=12)
        
    plt.title(f'Hand Position Prediction (CNN-LSTM) (Accuracy: {accuracy_val:.2f}%)', fontsize=16)
    plt.xlabel(x_label, fontsize=14); plt.ylabel('Box Number', fontsize=14); plt.legend(fontsize=12); plt.grid(True, alpha=0.7); plt.tight_layout()
    try: plt.savefig(output_filename, format='eps', dpi=300, bbox_inches='tight'); print(f"Time series plot: {os.path.abspath(output_filename)}")
    except Exception as e: print(f"Visualization saving error: {e}")
    plt.show(block=False); plt.pause(1); return plt


# --- Main Script ---
print(f"--- CNN-LSTM (X-Y Heatmap + Velocity, Refined Regional GT) Training ---")
print(f"Sequence Length: {SEQUENCE_LENGTH}, Number of Heatmap Channels: {NUM_CHANNELS_XY_VEL}")

# 1. Load Data
data_paths = discover_data_files(BASE_DATA_DIRECTORY)
if not data_paths: raise SystemExit("Data file not found.")
df_point_cloud_with_tof_loaded = load_and_preprocess_data(data_paths, FRAME_ID_COLUMN)
if df_point_cloud_with_tof_loaded.empty: raise SystemExit("Loaded data is empty.")

# 2. Prepare Heatmap Sequences, Labels, and Metadata for ALL DATA
print("\n--- Preparing Heatmap Sequences, Labels, and Metadata for All Data ---")
X_sequences_all, y_labels_all_refined, df_meta_all_sequences = create_sequences_labels_and_metadata(
    df_point_cloud_with_tof_loaded, FRAME_ID_COLUMN,
    threshold_inside_tof=THRESHOLD_INSIDE_TOF,
    region_window_frames=REGION_WINDOW_SIZE_FRAMES,
    sequence_length=SEQUENCE_LENGTH
)

if X_sequences_all.size == 0 or y_labels_all_refined.size == 0:
    raise SystemExit("Sequence or labels could not be created.")

# 3. Data Splitting (Train/Test) - with metadata
print("\n--- Data Splitting (Train/Test) ---")
test_set_fraction = 0.2; random_state_split = 42
stratify_opt_seq = y_labels_all_refined if len(np.unique(y_labels_all_refined)) > 1 else None

X_train_seq, X_test_seq, \
y_train_labels_refined, y_test_labels_refined, \
df_meta_train_seq, df_meta_test_seq = train_test_split(
    X_sequences_all, y_labels_all_refined, df_meta_all_sequences,
    test_size=test_set_fraction, random_state=random_state_split, stratify=stratify_opt_seq
)
print(f"X_train (sequences): {X_train_seq.shape}, y_train (GT refined): {y_train_labels_refined.shape}, df_meta_train_seq: {df_meta_train_seq.shape}")
print(f"X_test (sequences): {X_test_seq.shape}, y_test (GT refined): {y_test_labels_refined.shape}, df_meta_test_seq: {df_meta_test_seq.shape}")
print(f"Training set GT (refined) label distribution: {pd.Series(y_train_labels_refined).value_counts(normalize=True).sort_index()}")
print(f"Test set GT (refined) label distribution: {pd.Series(y_test_labels_refined).value_counts(normalize=True).sort_index()}")


# 4. Creating CNN-LSTM Model
print("\n--- Creating CNN-LSTM Model ---")
input_shape_seq = (SEQUENCE_LENGTH, IMG_HEIGHT_XY, IMG_WIDTH_XY, NUM_CHANNELS_XY_VEL)
cnn_input_shape_single_frame = (IMG_HEIGHT_XY, IMG_WIDTH_XY, NUM_CHANNELS_XY_VEL)

cnn_base = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=cnn_input_shape_single_frame), BatchNormalization(), MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D((2,2)),
    Conv2D(128, (3,3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D((2,2)), Flatten()
], name="cnn_base")
video_input = Input(shape=input_shape_seq, name="video_input")
encoded_frames = TimeDistributed(cnn_base, name="time_distributed_cnn")(video_input)
encoded_sequence = GRU(128, dropout=0.3, recurrent_dropout=0.2, name="gru_layer")(encoded_frames) # GRU used
output_layer = Dense(NUM_CLASSES, activation='softmax', name="output_layer")(encoded_sequence)
model_cnnlstm = Model(inputs=video_input, outputs=output_layer, name="cnn_lstm_model")

lr_sched_cnnlstm = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=0.0005, decay_steps=8000, decay_rate=0.92)
model_cnnlstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_sched_cnnlstm),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_cnnlstm.summary()


# 5. Model Training
print("\n--- CNN-LSTM Model Training ---")
epochs_cnnlstm = 120; batch_cnnlstm = 32 # Epochs can be increased, rely on patience
unique_tr_labels_cnnlstm = np.unique(y_train_labels_refined)
class_weights_cnnlstm = None
if len(unique_tr_labels_cnnlstm) > 1:
    class_w_calc_cnnlstm = compute_class_weight('balanced', classes=unique_tr_labels_cnnlstm, y=y_train_labels_refined)
    class_weights_cnnlstm = dict(zip(unique_tr_labels_cnnlstm, class_w_calc_cnnlstm)); print(f"Class weights: {class_weights_cnnlstm}")

callbacks_cnnlstm = [
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=1),
    ModelCheckpoint('../models/best_cnnlstm_model_vel_v2.keras', monitor='val_accuracy', save_best_only=True, verbose=1, mode='max')
]
history_cnnlstm = model_cnnlstm.fit(
    X_train_seq, y_train_labels_refined,
    epochs=epochs_cnnlstm, batch_size=batch_cnnlstm,
    validation_data=(X_test_seq, y_test_labels_refined),
    callbacks=callbacks_cnnlstm, class_weight=class_weights_cnnlstm, verbose=1
)
best_model_cnnlstm_file = '../models/best_cnnlstm_model_vel_v2.keras'
if os.path.exists(best_model_cnnlstm_file):
    print(f"Loading best CNN-LSTM model '{best_model_cnnlstm_file}'...")
    model_cnnlstm = tf.keras.models.load_model(best_model_cnnlstm_file)

# 6. Model Evaluation
print("\n--- CNN-LSTM Model Evaluation (TEST SET - REFINED GT) ---")
loss_test_cnnlstm, acc_test_cnnlstm = model_cnnlstm.evaluate(X_test_seq, y_test_labels_refined, verbose=0)
acc_test_cnnlstm *= 100
print(f"CNN-LSTM Test Loss (Refined GT): {loss_test_cnnlstm:.4f}")
print(f"CNN-LSTM Test Accuracy (Refined GT): {acc_test_cnnlstm:.2f}%")

y_pred_proba_cnnlstm = model_cnnlstm.predict(X_test_seq)
y_pred_classes_cnnlstm = np.argmax(y_pred_proba_cnnlstm, axis=1)
class_labels_str_all = [f'Box {l}' if l != 0 else 'No Box' for l in range(NUM_CLASSES)]
present_labels_num_cnnlstm = sorted(list(set(y_test_labels_refined) | set(y_pred_classes_cnnlstm)))
report_target_names_cnnlstm = [class_labels_str_all[l] for l in present_labels_num_cnnlstm if l < NUM_CLASSES]

print("\nClassification Report (CNN-LSTM Test Set - Refined GT):")
try:
    class_report_cnnlstm = classification_report(
        y_test_labels_refined, y_pred_classes_cnnlstm,
        labels=present_labels_num_cnnlstm, target_names=report_target_names_cnnlstm, zero_division=0
    )
    print(class_report_cnnlstm)
    report_path_cnnlstm = 'classification_report_CNN_LSTM_vel_v2.txt'
    with open(report_path_cnnlstm, 'w') as f:
        f.write(f"CNN-LSTM Test Accuracy (Refined GT): {acc_test_cnnlstm:.2f}%\n\n{class_report_cnnlstm}")
    print(f"Report saved: {os.path.abspath(report_path_cnnlstm)}")
except Exception as e: print(f"Classification report error: {e}")

# 7. Plots (Will now work with df_meta_test_seq)
print("\n--- CNN-LSTM Plots ---")
plot_confusion_matrix_custom(y_test_labels_refined, y_pred_classes_cnnlstm, class_labels_str_all,
                             output_filename='confusion_matrix_CNN_LSTM_vel_v2.eps',
                             title='Confusion Matrix (CNN-LSTM - Refined GT)')

# Add predictions to df_meta_test_seq
df_meta_test_seq_copy = df_meta_test_seq.copy() # Plotting functions might modify the DataFrame.
if len(df_meta_test_seq_copy) == len(y_pred_classes_cnnlstm): # Size check
    df_meta_test_seq_copy['predicted_label_cnnlstm'] = y_pred_classes_cnnlstm
    # True labels should already be in df_meta_test_seq as 'ground_truth_label'
    # (check the 'ground_truth_label' column of the df coming from create_sequences_labels_and_metadata)
    # If 'ground_truth_label' does not exist, use y_test_labels_refined.
    if 'ground_truth_label' not in df_meta_test_seq_copy.columns:
         df_meta_test_seq_copy['ground_truth_label'] = y_test_labels_refined

    plot_per_experiment_accuracy_cnn(df_meta_test_seq_copy,
                                     target_col_true='ground_truth_label', # column name in df_meta_test_seq
                                     target_col_pred='predicted_label_cnnlstm',
                                     output_filename='per_exp_acc_CNN_LSTM_vel_v2.eps')

    visualize_boxes_over_time_cnn(df_meta_test_seq_copy, FRAME_ID_COLUMN, # FRAME_ID_COLUMN should be in df_meta_test_seq
                                  'ground_truth_label', 'predicted_label_cnnlstm', acc_test_cnnlstm,
                                  output_filename='boxes_time_CNN_LSTM_vel_v2.eps')
else:
    print(f"WARNING: Size mismatch between df_meta_test_seq ({len(df_meta_test_seq_copy)}) and CNN-LSTM predictions ({len(y_pred_classes_cnnlstm)}). Skipping time series plots.")


# 8. Saving Model and Configuration
model_dir_cnnlstm = 'saved_model_cnnlstm_vel_v2'
os.makedirs(model_dir_cnnlstm, exist_ok=True)
config_path_cnnlstm = os.path.join(model_dir_cnnlstm, 'config_cnnlstm_vel_v2.txt')
with open(config_path_cnnlstm, 'w') as f:
    f.write(f"MODEL_TYPE=CNN_LSTM_with_Velocity_v2\n")
    f.write(f"BASE_DATA_DIRECTORY={BASE_DATA_DIRECTORY}\nFRAME_ID_COLUMN={FRAME_ID_COLUMN}\nNUM_CLASSES={NUM_CLASSES}\n")
    f.write(f"THRESHOLD_INSIDE_TOF={THRESHOLD_INSIDE_TOF}\nREGION_WINDOW_SIZE_FRAMES={REGION_WINDOW_SIZE_FRAMES}\n")
    f.write(f"SEQUENCE_LENGTH={SEQUENCE_LENGTH}\n")
    f.write(f"HEATMAP_TYPE=X-Y_TopView_SNR_and_Velocity\nX_MIN_METERS={X_MIN_METERS}, X_MAX_METERS={X_MAX_METERS}\n")
    f.write(f"Y_MIN_METERS={Y_MIN_METERS}, Y_MAX_METERS={Y_MAX_METERS}\n")
    f.write(f"X_BINS={X_BINS}, Y_BINS={Y_BINS}, NUM_CHANNELS_XY_VEL={NUM_CHANNELS_XY_VEL}\n")
    f.write(f"Test Set Fraction: {test_set_fraction}\nEpochs: {epochs_cnnlstm}, Batch Size: {batch_cnnlstm}\n")
    f.write(f"Final Test Accuracy (Refined GT): {acc_test_cnnlstm:.2f}%\n")
print(f"Configuration saved: {config_path_cnnlstm}")
print(f"Best CNN-LSTM model saved as '{os.path.abspath(best_model_cnnlstm_file)}'.")
plt.show(block=True) # Show all plots at the end of the script and keep them open
print("\n--- CNN-LSTM (X-Y Heatmap + Velocity, Refined Regional GT) Training Script Completed ---")